# Cross-Axis Analysis: Statistical Descriptor-Bit Bridge

This notebook implements the robust statistical bridge between Axis 1 and Axis 2:

1. Compute a point-biserial (Pearson) correlation matrix between ECFP4 bits (2048) and RDKit descriptors (~201).
2. For each bit, select its best-matching descriptor by maximum absolute correlation.
3. Join this with Axis 1 linear probe $R^2$ and Axis 2 per-bit AUROC, then plot $(R^2, AUROC)$ and report Spearman correlation.

This avoids structural interpretation and SMARTS recovery entirely; it uses direct statistical co-occurrence.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import AllChem

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# -----------------------------
# Config
# -----------------------------
ROOT = Path('..')
DATA_DIR = ROOT / 'data' / 'processed'
RESULTS_DIR = ROOT / 'results'
OUT_DIR = RESULTS_DIR / 'cross_axis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Core inputs (detected from existing thesis outputs)
DESCRIPTOR_TABLE_PATH = DATA_DIR / 'massspecgym_complete' / 'all_rdkit_descriptors.parquet'
AXIS1_R2_PATH = RESULTS_DIR / 'indicators' / 'probe_indicator_merged.csv'
AXIS2_AUROC_PATH = RESULTS_DIR / 'per_bit_analysis' / 'per_bit_auroc' / 'auroc_comparison.csv'

# Optional: precomputed ECFP4 matrix aligned row-wise with the descriptor table after filtering.
# If None or missing, ECFP4 is computed from SMILES directly in this notebook.
ECFP4_MATRIX_PATH = None

# Optional: restrict to training molecules only by providing a table with smiles/inchikey.
# If None, all molecules in the descriptor table are used.
TRAIN_MOLECULES_PATH = None

# Fingerprint parameters
N_BITS = 2048
RADIUS = 2

print('Using paths:')
print(f'  Descriptor table: {DESCRIPTOR_TABLE_PATH}')
print(f'  Axis 1 R2 table:  {AXIS1_R2_PATH}')
print(f'  Axis 2 AUROC:     {AXIS2_AUROC_PATH}')
print(f'  Output dir:       {OUT_DIR}')

In [ ]:
# -----------------------------
# Load Axis 1 and Axis 2 summary tables
# -----------------------------
axis1 = pd.read_csv(AXIS1_R2_PATH)
axis2 = pd.read_csv(AXIS2_AUROC_PATH)

required_axis1 = {'descriptor', 'r2_linear'}
required_axis2 = {'bit_index', 'auroc_val'}

missing_a1 = required_axis1 - set(axis1.columns)
missing_a2 = required_axis2 - set(axis2.columns)
if missing_a1:
    raise ValueError(f'Missing Axis 1 columns: {missing_a1}')
if missing_a2:
    raise ValueError(f'Missing Axis 2 columns: {missing_a2}')

# Keep one row per descriptor, prefer non-null r2_linear
axis1 = axis1[['descriptor', 'r2_linear']].dropna(subset=['descriptor']).copy()
axis1 = axis1.sort_values('descriptor').drop_duplicates(subset='descriptor', keep='first')

# Keep one row per bit index
axis2 = axis2[['bit_index', 'auroc_val']].copy()
axis2 = axis2.dropna(subset=['bit_index']).copy()
axis2['bit_index'] = axis2['bit_index'].astype(int)
axis2 = axis2.sort_values('bit_index').drop_duplicates(subset='bit_index', keep='first')

print(f'Axis 1 descriptors with r2_linear: {len(axis1)}')
print(f'Axis 2 bits with auroc_val:        {len(axis2)}')

In [ ]:
# -----------------------------
# Load descriptor table and align molecules
# -----------------------------
df_desc = pd.read_parquet(DESCRIPTOR_TABLE_PATH)

# Identify molecule key and smiles column
key_col = None
for candidate in ['inchikey', 'InChIKey', 'smiles', 'SMILES']:
    if candidate in df_desc.columns:
        key_col = candidate
        break
if key_col is None:
    raise ValueError('No molecule key found in descriptor table (expected inchikey or smiles).')

smiles_col = None
for candidate in ['smiles', 'SMILES']:
    if candidate in df_desc.columns:
        smiles_col = candidate
        break
if smiles_col is None and ECFP4_MATRIX_PATH is None:
    raise ValueError('No SMILES column found; needed to compute ECFP4 when ECFP4_MATRIX_PATH is not provided.')

# Keep only descriptors that are present in Axis 1 table
axis1_descriptor_order = axis1['descriptor'].tolist()
descriptor_cols = [d for d in axis1_descriptor_order if d in df_desc.columns]

if len(descriptor_cols) == 0:
    raise ValueError('No overlapping descriptor columns between descriptor table and Axis 1 results.')

missing_desc = [d for d in axis1_descriptor_order if d not in df_desc.columns]
if missing_desc:
    print(f'Warning: {len(missing_desc)} Axis 1 descriptors not found in descriptor table. They will be excluded.')

keep_cols = [key_col] + descriptor_cols
if smiles_col and smiles_col not in keep_cols:
    keep_cols.append(smiles_col)

df = df_desc[keep_cols].copy()

# Optional training-set restriction
if TRAIN_MOLECULES_PATH is not None and Path(TRAIN_MOLECULES_PATH).exists():
    train_df = pd.read_parquet(TRAIN_MOLECULES_PATH) if str(TRAIN_MOLECULES_PATH).endswith('.parquet') else pd.read_csv(TRAIN_MOLECULES_PATH)
    train_key = None
    for candidate in [key_col, 'inchikey', 'InChIKey', 'smiles', 'SMILES']:
        if candidate in train_df.columns:
            train_key = candidate
            break
    if train_key is None:
        raise ValueError('TRAIN_MOLECULES_PATH provided, but no key column found for merge.')

    train_keys = set(train_df[train_key].dropna().astype(str).unique())
    before = len(df)
    df = df[df[key_col].astype(str).isin(train_keys)].copy()
    print(f'Restricted to training molecules: {before} -> {len(df)}')
else:
    print('TRAIN_MOLECULES_PATH not provided. Using all molecules in descriptor table.')

# Drop duplicates by key and rows with missing descriptor values
df = df.drop_duplicates(subset=[key_col], keep='first').reset_index(drop=True)
df = df.dropna(subset=descriptor_cols).reset_index(drop=True)

print(f'Molecule rows after filtering: {len(df)}')
print(f'Number of descriptor columns used: {len(descriptor_cols)}')

In [ ]:
# -----------------------------
# Build aligned matrices
# X_desc: (N, D) continuous descriptor matrix
# X_bits: (N, 2048) binary ECFP4 matrix
# -----------------------------
X_desc = df[descriptor_cols].to_numpy(dtype=np.float32)
N = X_desc.shape[0]
D = X_desc.shape[1]

if ECFP4_MATRIX_PATH is not None and Path(ECFP4_MATRIX_PATH).exists():
    X_bits = np.load(ECFP4_MATRIX_PATH)
    if X_bits.shape[0] != N:
        raise ValueError(f'ECFP4 rows ({X_bits.shape[0]}) do not match descriptor rows ({N}).')
    if X_bits.shape[1] != N_BITS:
        raise ValueError(f'ECFP4 bits ({X_bits.shape[1]}) do not match expected N_BITS ({N_BITS}).')
    X_bits = X_bits.astype(np.float32)
    print(f'Loaded ECFP4 from file: {X_bits.shape}')
else:
    if smiles_col is None:
        raise ValueError('Cannot compute ECFP4 without SMILES column.')

    smiles = df[smiles_col].astype(str).tolist()
    X_bits = np.zeros((N, N_BITS), dtype=np.float32)

    failed = 0
    for i, s in enumerate(smiles):
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            failed += 1
            continue

        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=RADIUS, nBits=N_BITS)
        arr = np.zeros((N_BITS,), dtype=np.int8)
        # RDKit explicit bit vector supports conversion via list of on bits
        on_bits = list(fp.GetOnBits())
        arr[on_bits] = 1
        X_bits[i] = arr

    print(f'Computed ECFP4 matrix: {X_bits.shape}; failed SMILES: {failed}')

print(f'X_desc shape: {X_desc.shape}')
print(f'X_bits shape: {X_bits.shape}')

In [ ]:
# -----------------------------
# Computation 1: correlation matrix (point-biserial / Pearson)
# corr_matrix shape: (2048, D)
# -----------------------------
# Center columns
B = X_bits
Y = X_desc

B_mean = B.mean(axis=0, keepdims=True)
Y_mean = Y.mean(axis=0, keepdims=True)

B_center = B - B_mean
Y_center = Y - Y_mean

# Sample std with ddof=1
B_std = B_center.std(axis=0, ddof=1)
Y_std = Y_center.std(axis=0, ddof=1)

# Covariance matrix: (bits x descriptors)
cov = (B_center.T @ Y_center) / max(B.shape[0] - 1, 1)

den = np.outer(B_std, Y_std)
corr_matrix = np.full_like(cov, np.nan, dtype=np.float32)
valid = den > 0
corr_matrix[valid] = (cov[valid] / den[valid]).astype(np.float32)

print(f'corr_matrix shape: {corr_matrix.shape}')
print(f'finite entries: {np.isfinite(corr_matrix).sum():,} / {corr_matrix.size:,}')

In [ ]:
# For each bit, find best descriptor by absolute correlation
abs_corr = np.abs(corr_matrix)
all_nan_rows = np.all(~np.isfinite(abs_corr), axis=1)

best_desc_idx = np.zeros((N_BITS,), dtype=np.int32)
best_desc_idx[~all_nan_rows] = np.nanargmax(abs_corr[~all_nan_rows], axis=1)
best_desc_idx[all_nan_rows] = -1

best_corr = np.full((N_BITS,), np.nan, dtype=np.float32)
best_abs_corr = np.full((N_BITS,), np.nan, dtype=np.float32)

ok = best_desc_idx >= 0
bit_rows = np.arange(N_BITS)[ok]
desc_cols = best_desc_idx[ok]
best_corr[ok] = corr_matrix[bit_rows, desc_cols]
best_abs_corr[ok] = abs_corr[bit_rows, desc_cols]

print(f'Bits with valid best descriptor: {ok.sum()} / {N_BITS}')

In [ ]:
# -----------------------------
# Computation 2: Join Axis 1 R2 and Axis 2 AUROC via best descriptor per bit
# -----------------------------
axis1_map = axis1.set_index('descriptor')['r2_linear']

best_desc_name = np.array([descriptor_cols[i] if i >= 0 else None for i in best_desc_idx], dtype=object)
best_desc_r2 = np.array([axis1_map.get(name, np.nan) if name is not None else np.nan for name in best_desc_name], dtype=np.float32)

bridge = pd.DataFrame({
    'bit_index': np.arange(N_BITS, dtype=int),
    'best_descriptor_idx': best_desc_idx,
    'best_descriptor': best_desc_name,
    'best_corr': best_corr,
    'best_abs_corr': best_abs_corr,
    'r2_linear_best_descriptor': best_desc_r2,
})

bridge = bridge.merge(axis2[['bit_index', 'auroc_val']], on='bit_index', how='left')

plot_df = bridge.dropna(subset=['r2_linear_best_descriptor', 'auroc_val']).copy()

spearman_rho = plot_df['r2_linear_best_descriptor'].corr(plot_df['auroc_val'], method='spearman')
pearson_r = plot_df['r2_linear_best_descriptor'].corr(plot_df['auroc_val'], method='pearson')

print(f'Bridge points: {len(plot_df)}')
print(f'Spearman rho (R2 vs AUROC): {spearman_rho:.4f}')
print(f'Pearson r (R2 vs AUROC):    {pearson_r:.4f}')

In [ ]:
# Save outputs
np.save(OUT_DIR / 'corr_matrix_bits_x_descriptors.npy', corr_matrix)
np.save(OUT_DIR / 'best_descriptor_idx_per_bit.npy', best_desc_idx)
bridge.to_csv(OUT_DIR / 'bit_descriptor_bridge.csv', index=False)

summary = pd.DataFrame([{
    'n_molecules': int(N),
    'n_bits': int(N_BITS),
    'n_descriptors_used': int(D),
    'n_bridge_points': int(len(plot_df)),
    'spearman_rho_r2_vs_auroc': float(spearman_rho),
    'pearson_r_r2_vs_auroc': float(pearson_r),
}])
summary.to_csv(OUT_DIR / 'cross_axis_summary.csv', index=False)

print('Saved:')
print(OUT_DIR / 'corr_matrix_bits_x_descriptors.npy')
print(OUT_DIR / 'best_descriptor_idx_per_bit.npy')
print(OUT_DIR / 'bit_descriptor_bridge.csv')
print(OUT_DIR / 'cross_axis_summary.csv')

In [ ]:
# Scatter plot: Axis 1 descriptor linear R2 vs Axis 2 per-bit AUROC
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    plot_df['r2_linear_best_descriptor'],
    plot_df['auroc_val'],
    s=14,
    alpha=0.35,
    edgecolors='none',
    color='#1f77b4'
)

ax.set_xlabel('Axis 1 linear probe $R^2$ (best-matching descriptor)')
ax.set_ylabel('Axis 2 per-bit AUROC (validation)')
ax.set_title('Cross-axis statistical bridge: descriptor-bit matching via correlation')

txt = f'Spearman rho = {spearman_rho:.3f}\nPearson r = {pearson_r:.3f}\nN = {len(plot_df)} bits'
ax.text(0.03, 0.97, txt, transform=ax.transAxes, va='top', ha='left', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor='0.8'))

fig.tight_layout()
png_path = OUT_DIR / 'cross_axis_r2_vs_auroc_scatter.png'
pdf_path = OUT_DIR / 'cross_axis_r2_vs_auroc_scatter.pdf'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')
plt.show()

print(f'Saved figure: {png_path}')
print(f'Saved figure: {pdf_path}')

## Notes

- Point-biserial correlation is equivalent to Pearson correlation when one variable is binary.
- This method sidesteps hashed-bit structural ambiguity by using statistical co-occurrence only.
- If you want strict training-only chemistry, set `TRAIN_MOLECULES_PATH` to a training split table containing `smiles` or `inchikey`.